# 02. Silver to Gold - Architecture Splitting

**Goal:** Transform the structurally pure Silver layer (`loans_silver.parquet`) into a business-ready dataset. 

**CRITICAL ARCHITECTURE DECISION:**
A single dataset cannot serve both Business Intelligence (BI) and Machine Learning (ML) without causing severe data leakage or statistical bias. 
- BI requires hindsight data (e.g., total recoveries) and bounded charts (Winsorized outliers).
- ML requires strictly foresight data (features known at origination) and untampered distributions before train/test splits.

Therefore, this notebook branches at the end to export two distinct Gold artifacts: `loans_gold_bi.parquet` and `loans_gold_ml.parquet`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
# ── Load Silver Data ───────────────────────────────────────
df_clean = pd.read_parquet('../data/silver/loans_silver.parquet')
df_clean = df_clean.rename(columns={'id': 'loan_id'})
df_clean['loan_id'] = df_clean['loan_id'].astype('int64')
print(f"Initial Shape from Silver: {df_clean.shape}")

Initial Shape from Silver: (2260668, 86)


## 1. Target Definition & Leakage Drop
We focus strictly on defining the target here. 

> **Note on Proxy Censoring:** We use `Late (31-120 days)` as a proxy for Default. The materiality of this proxy is quantified below to confirm it does not introduce excessive label noise.

**CRITICAL:** We now preserve `Current` loans in the dataset (with `default_flag = NaN`) so they can flow into the BI Dashboard. The ML pipeline will strictly filter them out later.

In [3]:
import numpy as np
default_status = ['Charged Off', 'Late (31-120 days)', 'Default', 'Does not meet the credit policy. Status:Charged Off']
paid_status = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']

# We DO NOT drop "Current" loans here. BI needs them to show portfolio volume.
# We create the target flag: 1 = Default, 0 = Paid, NaN = Current/InProgress
df_clean['default_flag'] = np.where(df_clean['loan_status'].isin(default_status), 1, 
                           np.where(df_clean['loan_status'].isin(paid_status), 0, np.nan))

print(f"Total rows (including Current): {df_clean.shape[0]:,}")
print(f"\nDefault flag distribution (NaN = Current):")
print(df_clean['default_flag'].value_counts(dropna=False))

# Proxy materiality: what % of default_flag=1 is truly terminal vs proxy?
proxy_check = df_clean[df_clean['default_flag']==1]['loan_status'].value_counts()
proxy_pct = proxy_check.get('Late (31-120 days)', 0) / proxy_check.sum() * 100
print(f"\nProxy materiality: Late (31-120 days) = {proxy_pct:.1f}% of all defaults")
print(proxy_check)


Total rows (including Current): 2,260,668

Default flag distribution (NaN = Current):
default_flag
0.0    1078739
NaN     891102
1.0     290827
Name: count, dtype: int64

Proxy materiality: Late (31-120 days) = 7.4% of all defaults
loan_status
Charged Off                                            268559
Late (31-120 days)                                      21467
Does not meet the credit policy. Status:Charged Off       761
Default                                                    40
Current                                                     0
Does not meet the credit policy. Status:Fully Paid          0
Fully Paid                                                  0
In Grace Period                                             0
Late (16-30 days)                                           0
Name: count, dtype: int64


## 2. Feature Selection & Dimensionality Reduction
We drop redundant, administrative, and overly granular metrics. 
> **Architecture Note:** `hardship_flag`, `debt_settlement_flag`, `out_prncp_inv`, and `total_pymnt_inv` do not appear here because they were **already purged in the Bronze to Silver step**. We only keep post-origination fields (`recoveries`, `total_pymnt`) here initially, but we will branch them out before the ML export.\n>\n> **Architecture Note (The Great Purge):** Highly granular credit bureau features (`num_tl_90g_dpd_24m`, `pct_tl_nvr_dlq`, etc.) are dropped here. Their correlation and feature importance should be validated during the downstream modeling phase to ensure no critical predictive signal was lost.

In [4]:
cols_to_drop = [
    # Irrelevant or administrative features
    'policy_code', 'pymnt_plan', 'initial_list_status', 
    'hardship_flag', 'disbursement_method', 'debt_settlement_flag', 
    
    # Redundant FICO features (we will engineer a single fico_score later)
    'last_fico_range_low', 'last_fico_range_high',
    
    # Time since account opening (Too granular)
    'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 
    'mo_sin_rcnt_tl', 
    
    # Granular account counts
    'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 
    'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 
    'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 
    'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    
    # Redundant balances
    'tot_coll_amt', 'total_rev_hi_lim', 'avg_cur_bal', 'bc_open_to_buy', 
    'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit'
]

# Ensure we only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]

df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Purge complete: {df_clean.shape[1]} remaining columns.")

Purge complete: 52 remaining columns.


## 3. Feature Transformation & Linear Safety
> **Linear Safety (`emp_length`)**: Null values in employment length are flagged in a separate binary column (`emp_length_missing`), and the raw column is kept as NaN to defer imputation to the downstream ML pipeline. This prevents linear models (like Logistic Regression) from inferring a false continuous relationship between "Unknown" (-1) and "Junior" (0).
>
> **Temporal Risk (`issue_d`)**: This dataset preserves `issue_d` as a raw datetime. ML models **must not** ingest this raw date directly, as they will learn vintage/macroeconomic cycles rather than true borrower risk.

In [5]:
# 1. Average the FICO score 
df_clean['fico_score'] = (df_clean['fico_range_low'] + df_clean['fico_range_high']) / 2
df_clean = df_clean.drop(columns=['fico_range_low', 'fico_range_high'])

# 2. term: "36 months" → 36
if df_clean['term'].dtype == 'O' or isinstance(df_clean['term'].dtype, pd.CategoricalDtype):
    assert df_clean['term'].notna().all(), 'NaN found in term column'
    df_clean['term'] = df_clean['term'].astype(str).str.extract(r'(\d+)')[0].astype(int)

# 3. emp_length mapping & linear safety
# In Silver, nulls were filled with "Not Disclosed"
df_clean["emp_length_missing"] = (df_clean["emp_length"] == "Not Disclosed").astype(int)
assert df_clean['emp_length'].isna().sum() == 0, "Quedan NaN reales en emp_length, la promesa de Silver no se cumplio"
emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
    "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
    "8 years": 8, "9 years": 9, "10+ years": 10, "Not Disclosed": np.nan
}
assert set(df_clean["emp_length"].dropna().unique()).issubset(set(emp_map.keys())), "emp_map fail: unrecognized strings found"
df_clean["emp_length"] = df_clean["emp_length"].map(emp_map)
# Note: Imputation deferred to downstream ML pipeline to prevent distribution leakage.

# 4. issue_d & earliest_cr_line -> credit history in years
df_clean["issue_d"] = pd.to_datetime(df_clean["issue_d"], format="%b-%Y", errors='coerce')
invalid_issue_d = df_clean["issue_d"].isna().sum()
if invalid_issue_d > 0:
    print(f"Dropped {invalid_issue_d} rows due to invalid issue_d")
    df_clean = df_clean.dropna(subset=["issue_d"])
assert df_clean["issue_d"].notna().all(), "Date parsing failed for issue_d"
df_clean["earliest_cr_line"] = pd.to_datetime(df_clean["earliest_cr_line"], format="%b-%Y", errors='coerce')
df_clean["credit_history_missing"] = df_clean["earliest_cr_line"].isna().astype(int)
df_clean["credit_history_years"] = (df_clean["issue_d"] - df_clean["earliest_cr_line"]).dt.days / 365.25
assert (df_clean["credit_history_years"].dropna() >= 0).all(), "Negative credit history found"
df_clean = df_clean.drop(columns=["earliest_cr_line"])

# 5. int_rate -> convert to decimal
if df_clean["int_rate"].max() > 1:
    df_clean["int_rate"] = df_clean["int_rate"] / 100

print("Transformations done ✓")

Transformations done ✓


## 4. Missing Data Imputation & Bias Check

We bin recency metrics into distinct categories. 

**CRITICAL - OPTION A (Preserving the 2008 Crisis):** `tot_cur_bal` is structurally missing before 2012. Calling `.dropna()` on it would silently erase the 2007-2011 vintages (the financial crisis). Instead, we flag it (`tot_cur_bal_missing`) and defer imputation to the ML pipeline. The remaining true random nulls (like `dti`) are dropped.

In [6]:
def bin_mths(x):
    if pd.isna(x): return 1
    elif x >= 36: return 2        # >=3 Years Ago (boundary: 36 inclusive)
    elif x <= 12: return 4
    else: return 3
FIELD_INTRODUCED_YEAR = 2012
for field in ['mths_since_recent_inq', 'mths_since_recent_bc']:
    if field in df_clean.columns:
        structurally_missing = (df_clean['issue_d'].dt.year < FIELD_INTRODUCED_YEAR) & df_clean[field].isna()
        df_clean[f'{field}_cat'] = df_clean[field].apply(bin_mths)
        df_clean.loc[structurally_missing, f'{field}_cat'] = 0
        df_clean = df_clean.drop(columns=[field])

# 3. bc_util (bankcard utilization)
if 'bc_util' in df_clean.columns:
    df_clean['bc_util_missing'] = df_clean['bc_util'].isna().astype(int)
    # Note: Imputation deferred to downstream ML pipeline to prevent 0-conflation

# 4. OPTION A: Preserve tot_cur_bal history
df_clean['tot_cur_bal_missing'] = df_clean['tot_cur_bal'].isna().astype(int)
if 'mort_acc' in df_clean.columns:
    df_clean['mort_acc_missing'] = df_clean['mort_acc'].isna().astype(int)
if 'acc_open_past_24mths' in df_clean.columns:
    df_clean['acc_open_past_24mths_missing'] = df_clean['acc_open_past_24mths'].isna().astype(int)
# Note: Imputation is deferred to downstream ML pipelines to prevent distribution leakage.

# ── Bias Audit ──
print(f"Target mean BEFORE drop: {df_clean['default_flag'].mean():.4f}")
print("\nRow count by year BEFORE drop:")
print(df_clean.groupby(df_clean['issue_d'].dt.year).size())

print("\n--- Proof of Ordinality for bin_mths (Default Rate by Bucket) ---")
if 'mths_since_recent_bc_cat' in df_clean.columns:
    print(df_clean.groupby('mths_since_recent_bc_cat')['default_flag'].mean().round(4))
    print('\n[AUDIT] Validating bin_mths bias across pre/post 2012 cohorts:')
    print(df_clean.groupby([df_clean['issue_d'].dt.year >= 2012, 'mths_since_recent_bc_cat'])['default_flag'].mean().round(4))
if 'mths_since_recent_inq_cat' in df_clean.columns:
    print(df_clean.groupby('mths_since_recent_inq_cat')['default_flag'].mean().round(4))

# Check remaining nulls before branching
print("\n--- Remaining Nulls in Shared DataFrame ---")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0].sort_values(ascending=False))

# Note: The final dropna() is now deferred to the ML and BI branches separately to avoid backdoor leakage.


Target mean BEFORE drop: 0.2123

Row count by year BEFORE drop:
issue_d
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134814
2014    235629
2015    421095
2016    434407
2017    443579
2018    495242
dtype: int64

--- Proof of Ordinality for bin_mths (Default Rate by Bucket) ---
mths_since_recent_bc_cat
0    0.1512
1    0.2052
2    0.1720
3    0.2097
4    0.2349
Name: default_flag, dtype: float64

[AUDIT] Validating bin_mths bias across pre/post 2012 cohorts:
issue_d  mths_since_recent_bc_cat
False    0                           0.1512
True     1                           0.2052
         2                           0.1720
         3                           0.2097
         4                           0.2349
Name: default_flag, dtype: float64
mths_since_recent_inq_cat
0    0.1512
1    0.1616
3    0.1868
4    0.2273
Name: default_flag, dtype: float64

--- Remaining Nulls in Shared DataFrame ---
default_flag            891102
emp_length

## 5. Architectural Split (ML vs BI)

**Path A (Machine Learning):** 
Extract strictly pre-origination features. **No global outlier capping** is performed here; capping percentiles must be calculated solely on the Train split in downstream pipelines to prevent distribution leakage.
> **Modeling Note on `int_rate` and `sub_grade`:** These are outputs of LendingClub's proprietary risk model. Using them to predict default is predicting LC's model, not the raw borrower risk. Downstream ML flows should run **two** experiments: one including them (Benchmark) and one excluding them (Independent Risk Model).

**Path B (Business Intelligence):**
Retain post-origination leakage features (`recoveries`, etc.) and explicitly cap extreme outliers (Winsorize) to stabilize Executive Dashboards.

> **Data Engineering Note:** Pandas lacks a vectorized `DATEADD` function for exact calendar month arithmetic. To avoid the severe performance penalty of `.apply()`, we leverage domain knowledge (terms are strictly 36 or 60 months) to vectorize the offset calculation in two fast operations. In a real-time production environment (PySpark/SQL), this would be replaced by a native `DATEADD(month, term, issue_d)`.

In [7]:
import os
os.makedirs('../data/gold', exist_ok=True)

# ==========================================
# PATH A: Machine Learning (Strict Foresight)
# ==========================================
leakage_cols = [
    'out_prncp', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 
    'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d'
]

df_ml = df_clean.drop(columns=[c for c in leakage_cols if c in df_clean.columns]).copy()

sospechosas = [c for c in df_clean.columns if any(k in c.lower() for k in ['hardship', 'settlement', 'pymnt_inv', 'out_prncp_inv', 'next_pymnt'])]
print(f"\nLeakage columns check (should be empty if purged in Bronze): {sospechosas}")

# --- CRITICAL FIX: Maturity Filter (Right-Censoring) ---
df_ml = df_ml.dropna(subset=['default_flag']).copy()
# Removed maturity filter for resolved loans (Issue #2)

# --- CRITICAL FIX: Branch-Specific DropNA ---
# We drop remaining nulls ONLY AFTER leakage columns are removed.
rows_before_ml = df_ml.shape[0]
allowed_nulls = ['tot_cur_bal', 'emp_length', 'bc_util', 'credit_history_years', 'mort_acc', 'acc_open_past_24mths']
subset_cols = [c for c in df_ml.columns if c not in allowed_nulls]
df_ml = df_ml.dropna(subset=subset_cols)
rows_after_ml = df_ml.shape[0]
print(f"ML Branch: Dropped {rows_before_ml - rows_after_ml:,} rows due to true MCAR nulls (e.g. dti).")

# Drop ML Leakage (loan_status, issue_d)
ml_drop_cols = ['loan_status', 'issue_d']
df_ml = df_ml.drop(columns=[c for c in ml_drop_cols if c in df_ml.columns])
df_ml['default_flag'] = df_ml['default_flag'].astype(int)

df_ml.to_parquet('../data/gold/loans_gold_ml.parquet', index=False)
print(f"⭐ ML Pipeline exported: {df_ml.shape[1]} features, strictly mature cohorts only.")
print(f"Target mean en ML (cohortes purgadas): {df_ml['default_flag'].mean():.4f}")
print(f"Filas ML final: {df_ml.shape}")

# ==========================================
# PATH B: Business Intelligence (Hindsight)
# ==========================================
df_bi = df_clean.copy()
# Removed maturity filter for resolved loans to prevent truncating the dataset
df_bi = df_bi.copy()

# BI Branch DropNA
rows_before_bi = df_bi.shape[0]
allowed_nulls_bi = ['tot_cur_bal', 'emp_length', 'default_flag', 'bc_util', 'last_pymnt_d', 'last_credit_pull_d', 'credit_history_years', 'mort_acc', 'acc_open_past_24mths']
subset_cols_bi = [c for c in df_bi.columns if c not in allowed_nulls_bi]
df_bi = df_bi.dropna(subset=subset_cols_bi)
rows_after_bi = df_bi.shape[0]
# Note (Auditor): This drops ~3,510 rows (0.16% of total) due to nulls in non-exempt columns.
# This is immaterial for business volume tracking, so we accept the minor loss of 'Current' rows.
print(f"BI Branch: Dropped {rows_before_bi - rows_after_bi:,} rows.")

# BI-Specific Winsorizing by Year (adjusts for inflation/temporal drift)
import numpy as np
def winsorize_by_year(df, col, q, min_group_size=5000):
    if col not in df.columns: return
    year_col = df['issue_d'].dt.year
    counts = year_col.value_counts()
    valid_years = np.array(sorted(counts[counts >= min_group_size].index))
    if valid_years.size == 0:
        df[col] = df[col].clip(upper=df[col].quantile(q))
        return
    def nearest_valid(y):
        return valid_years[np.abs(valid_years - y).argmin()]
    year_bucket = year_col.apply(nearest_valid)
    caps = df.groupby(year_bucket)[col].transform(lambda x: x.quantile(q))
    df[col] = df[col].clip(upper=caps)

winsorize_by_year(df_bi, 'dti', 0.99)
winsorize_by_year(df_bi, 'annual_inc', 0.999)
winsorize_by_year(df_bi, 'revol_bal', 0.999)

df_bi.to_parquet('../data/gold/loans_gold_bi.parquet', index=False)
print(f"⭐ BI Pipeline exported: {df_bi.shape[1]} features, retains Current loans for volume tracking.")
print(f"Filas BI final: {df_bi.shape}")




Leakage columns check (should be empty if purged in Bronze): []
ML Branch: Dropped 1,322 rows due to true MCAR nulls (e.g. dti).
⭐ ML Pipeline exported: 45 features, strictly mature cohorts only.
Target mean en ML (cohortes purgadas): 0.2123
Filas ML final: (1368244, 45)
BI Branch: Dropped 3,510 rows.
⭐ BI Pipeline exported: 57 features, retains Current loans for volume tracking.
Filas BI final: (2257158, 57)
